In [ ]:
# ==================================================================
#   CONFIGURATION - pick ONE job, then Save & Run All (Commit)
# ==================================================================
#
#   clip_emotion  fine-tune CLIP to predict the 7 emotions   (~2-3 h)
#   clip_overall  fine-tune CLIP to predict the score        (~2-3 h)
#   qwen4b        extract Qwen3-VL 4B hidden states          (~2-4 h)
#   qwen8b        extract Qwen3-VL 8B hidden states          (~4-6 h)
#
#   All four need the repaired ratings.csv. clip_* also need the images;
#   qwen* additionally download the model, so Internet must be On.

JOB = "clip_emotion"   # clip_emotion | clip_overall | qwen4b | qwen8b

# --- CLIP fine-tune settings (ignored by the qwen jobs) --------------
FOLDS      = [0, 1, 2, 3, 4]
EPOCHS     = 8
BATCH_SIZE = 32
LR         = 1e-5
SEED       = 42

# --- Qwen settings (ignored by the clip jobs) ------------------------
# Layer 15 follows Ryu & Yanaka, who report it as the representative
# layer across model sizes. Saving only the layers we might use keeps the
# output small; 'all' would be ~2 GB per model.
QWEN_SAVE_LAYERS = "15,17"

# Folder name of the uploaded dataset (None = search for it)
DATASET_SLUG = None

WORK_DIR = "/kaggle/working/piaa"

JOBS = {
    "clip_emotion": dict(kind="clip", target="emotion"),
    "clip_overall": dict(kind="clip", target="overall"),
    "qwen4b": dict(kind="qwen", model="Qwen/Qwen3-VL-4B-Instruct",
                   out="vlm4b_features.npz"),
    "qwen8b": dict(kind="qwen", model="Qwen/Qwen3-VL-8B-Instruct",
                   out="vlm_features.npz"),
}
assert JOB in JOBS, f"unknown JOB {JOB!r}; pick one of {list(JOBS)}"
CFG = JOBS[JOB]

print("=" * 65)
print("  JOB          :", JOB)
print("  kind         :", CFG["kind"])
if CFG["kind"] == "clip":
    print("  target       :", CFG["target"])
    print("  folds        :", FOLDS)
    print("  epochs/batch :", EPOCHS, "/", BATCH_SIZE)
else:
    print("  model        :", CFG["model"])
    print("  save layers  :", QWEN_SAVE_LAYERS)
print("=" * 65)

## How to run without keeping the tab open

**Save Version -> Save & Run All (Commit)** runs the whole notebook
server-side as a batch job. Close the browser; there is no idle timeout.
A session may run up to 12 h.

Before running: **Settings -> Accelerator -> GPU T4 x2**, **Internet -> On**.

Run one JOB per commit. The four jobs write to different filenames, so
they never overwrite each other.

In [ ]:
# --- Locate the uploaded dataset ------------------------------------
import os, sys, glob, shutil, time, subprocess

INPUT_ROOT = "/kaggle/input"

# Kaggle nests the dataset differently depending on how it was uploaded,
# so find the bundle by looking for its contents rather than a fixed path.
cands = []
if DATASET_SLUG:
    cands.append(os.path.join(INPUT_ROOT, DATASET_SLUG))
cands += sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "piaa_bundle"),
                         recursive=True))
cands += sorted(glob.glob(os.path.join(INPUT_ROOT, "*")))

SOURCE_DIR = None
for c in cands:
    for probe in (c, os.path.join(c, "piaa_bundle")):
        if os.path.exists(os.path.join(probe, "src")) and \
           os.path.exists(os.path.join(probe, "Dataset")):
            SOURCE_DIR = probe
            break
    if SOURCE_DIR:
        break

if SOURCE_DIR is None:
    print("[X] bundle not found. /kaggle/input holds:", os.listdir(INPUT_ROOT))
    raise SystemExit("set DATASET_SLUG to the dataset folder name")

print("[OK] SOURCE_DIR =", SOURCE_DIR)
print("     contents  :", sorted(os.listdir(SOURCE_DIR)))

In [ ]:
# --- Stage code + data on the writable disk -------------------------
# Checks the files themselves, not just WORK_DIR: the cleanup cell at the
# end of a previous run deletes the staged copies but leaves the folder,
# and a WORK_DIR-only check then skips re-staging and everything below
# fails with FileNotFoundError.
os.makedirs(WORK_DIR, exist_ok=True)

for name in ("src", "Dataset"):
    dst = os.path.join(WORK_DIR, name)
    need = not os.path.exists(os.path.join(WORK_DIR, "Dataset", "maked",
                                           "ratings.csv")) \
           if name == "Dataset" else \
           not os.path.exists(os.path.join(WORK_DIR, "src", "data",
                                           "finetune_clip_perfold.py"))
    if need:
        print(f"staging {name} ...", flush=True)
        shutil.copytree(os.path.join(SOURCE_DIR, name), dst, dirs_exist_ok=True)
    else:
        print(f"[OK] {name} already staged")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print("CWD:", os.getcwd())

In [ ]:
# --- Locate / stage the images --------------------------------------
# Kaggle unpacks an uploaded zip, and it unpacks nested zips too, so
# images/ may hold the archives or the already-extracted folders.
import zipfile

SRC_IMAGES = os.path.join(SOURCE_DIR, "images")
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def count_images(root):
    return sum(1 for r, _, fs in os.walk(root) for f in fs
               if f.lower().endswith(IMG_EXT))

IMAGES_DIR = os.path.join(WORK_DIR, "Dataset", "sample")
os.makedirs(IMAGES_DIR, exist_ok=True)

have = count_images(IMAGES_DIR)
if have:
    print("[OK] images already staged:", have)
else:
    zips = sorted(glob.glob(os.path.join(SRC_IMAGES, "*.zip")))
    n_src = count_images(SRC_IMAGES) if os.path.exists(SRC_IMAGES) else 0
    print("zip archives:", len(zips), "  extracted images in input:", n_src)
    t0 = time.time()
    if zips:
        for z in zips:
            print("  unpacking", os.path.basename(z), "...", flush=True)
            with zipfile.ZipFile(z) as f:
                f.extractall(IMAGES_DIR)
    elif n_src:
        print("  copying extracted images ...", flush=True)
        shutil.copytree(SRC_IMAGES, IMAGES_DIR, dirs_exist_ok=True)
    else:
        print("[X] no images under", SRC_IMAGES)
        for r, _, fs in os.walk(SOURCE_DIR):
            hits = [f for f in fs if f.lower().endswith(IMG_EXT)]
            if hits:
                print("   found images in", r, "e.g.", hits[0])
                break
    have = count_images(IMAGES_DIR)
    print("[OK] staged %d images in %.0fs" % (have, time.time() - t0))

print("image files available:", have)

In [ ]:
# --- Verify everything the job needs, before spending GPU time -------
import pandas as pd

RATINGS = "Dataset/maked/ratings.csv"
need = ["src/data/finetune_clip_perfold.py", "src/data/extract_vlm_features.py",
        RATINGS]
if CFG["kind"] == "clip":
    need += [f"Dataset/split_v4_10group/fold{k}/train_users.txt" for k in FOLDS]
    need += [f"Dataset/split_v4_10group/fold{k}/giaa_train_images.txt" for k in FOLDS]

ok = True
for f in need:
    e = os.path.exists(f)
    print(("[OK]  " if e else "[MISSING] ") + f)
    ok &= e

# The failure this guards against: 36 scenery filenames were stored as the
# spreadsheet error '#NAME?', and every image resolved through that column
# was silently skipped, leaving two backbones with 6490 images not 6526.
df = pd.read_csv(RATINGS)
broken = int((df["sample_file"].astype(str) == "#NAME?").sum())

def resolve(sf):
    sf = str(sf)
    return sf[:-4] + ".jpg" if sf.lower().endswith(".mp4") else sf

have_names = {f for r, _, fs in os.walk(IMAGES_DIR) for f in fs}
want = {resolve(s) for s in df["sample_file"].unique()}
missing = want - have_names

print()
print("'#NAME?' rows :", broken)
print("stimuli       :", len(want))
print("resolvable    :", len(want & have_names))
print("missing       :", len(missing))
if missing:
    print("  examples:", sorted(missing)[:5])

ok &= (broken == 0) and (len(missing) == 0)
print()
print("[OK] ready to run" if ok else "[X] STOP - fix the above first")
assert ok, "preflight failed"

In [ ]:
# --- GPU / versions --------------------------------------------------
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("pandas      ", pd.__version__)
print("cuda        ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print()
    print("[X] no GPU: Settings -> Accelerator -> GPU T4 x2")
    print("    on CPU this job takes ~30 h instead of ~2 h")
assert torch.cuda.is_available(), "turn the GPU on"

## Run

**clip_\***: each fold trains only on that fold's training-group users and
GIAA images, then extracts features for every image. Keeping the folds
separate is what makes the features leak-free -- one model fine-tuned on
all users inflated SROCC from 0.432 to 0.534 during development.

**qwen\***: one pass over every image, saving the chosen decoder layers.
Finished folds/files are skipped, so a re-run resumes instead of redoing.

In [ ]:
# --- Run -------------------------------------------------------------
FEATURES = "/kaggle/working/features"
os.makedirs(FEATURES, exist_ok=True)

def run(cmd, label):
    print("=" * 65)
    print("->", label)
    print("  " + " ".join(str(c) for c in cmd[1:]))
    print("=" * 65, flush=True)
    t0 = time.time()
    p = subprocess.run([str(c) for c in cmd], text=True)
    m, s = divmod(int(time.time() - t0), 60)
    print(f"   {label}: {m}m {s}s   exit={p.returncode}", flush=True)
    if p.returncode != 0:
        raise SystemExit(f"{label} failed")

t_all = time.time()

if CFG["kind"] == "clip":
    target = CFG["target"]
    outdir = os.path.join(FEATURES, f"clip_ftpf_{target}_v4_results")
    os.makedirs(outdir, exist_ok=True)
    for fold in FOLDS:
        out = os.path.join(outdir, f"clip_ftpf_{target}_v4_fold{fold}.npz")
        if os.path.exists(out):
            print(f"   fold {fold}: done already, skipping")
            continue
        run([sys.executable, "-u", "src/data/finetune_clip_perfold.py",
             "--fold", fold, "--target", target, "--v4",
             "--data_dir", "Dataset/maked",
             "--images_dir", IMAGES_DIR,
             "--v4_split_dir", "Dataset/split_v4_10group",
             "--out", out,
             "--epochs", EPOCHS, "--batch_size", BATCH_SIZE,
             "--lr", LR, "--seed", SEED], f"fold {fold}")
else:
    out = os.path.join(FEATURES, CFG["out"])
    if os.path.exists(out):
        print("   already extracted, skipping:", out)
    else:
        run([sys.executable, "-u", "src/data/extract_vlm_features.py",
             "--images_dir", IMAGES_DIR,
             "--ratings_csv", RATINGS,
             "--model", CFG["model"],
             "--save_layers", QWEN_SAVE_LAYERS,
             "--out", out], JOB)

h, rem = divmod(int(time.time() - t_all), 3600)
m, s = divmod(rem, 60)
print()
print(f"TOTAL: {h}h {m}m {s}s")

In [ ]:
# --- Verify the output before downloading ---------------------------
# 512 is the width CLIPModel.get_image_features() returns, i.e. after the
# visual projection -- the same space the frozen CLIP baseline uses, so the
# backbone comparison varies fine-tuning alone and not the feature width.
import numpy as np

if CFG["kind"] == "clip":
    target = CFG["target"]
    outdir = os.path.join(FEATURES, f"clip_ftpf_{target}_v4_results")
    sets = {}
    for fold in FOLDS:
        p = os.path.join(outdir, f"clip_ftpf_{target}_v4_fold{fold}.npz")
        if not os.path.exists(p):
            print(f"[MISSING] fold {fold}")
            continue
        z = np.load(p, allow_pickle=True)
        ids, feats = z["stimulus_ids"], z["features"]
        sets[fold] = set(map(str, ids))
        flag = "[OK] " if feats.shape[1] == 512 else "[X]  "
        print(f"{flag}fold {fold}: {feats.shape}  ({len(ids)} stimuli)")
    if sets:
        base = sets[min(sets)]
        same = all(s == base for s in sets.values())
        print()
        print(("[OK] " if same else "[X]  ") + "all folds cover the same stimuli")
        print(("[OK] " if len(base) == 6526 else "[X]  ") +
              f"stimulus count: {len(base)} (expected 6526)")
else:
    p = os.path.join(FEATURES, CFG["out"])
    z = np.load(p, allow_pickle=True)
    print("keys   :", list(z.keys()))
    print("ids    :", len(z["stimulus_ids"]), "(expected 6526)")
    print("layers :", z["layers"])
    print("LT     :", z["LT"].shape)
    print("size   : %.0f MB" % (os.path.getsize(p) / 1e6))

In [ ]:
# --- Clean up so the version's Output stays small --------------------
shutil.rmtree(IMAGES_DIR, ignore_errors=True)
shutil.rmtree(os.path.join(WORK_DIR, "Dataset"), ignore_errors=True)
shutil.rmtree(os.path.join(WORK_DIR, "src"), ignore_errors=True)

print("files to download from the Output tab:")
for r, _, fs in os.walk("/kaggle/working"):
    for f in sorted(fs):
        if f.endswith(".npz"):
            p = os.path.join(r, f)
            print("   %s  %.1f MB" % (os.path.relpath(p, "/kaggle/working"),
                                      os.path.getsize(p) / 1e6))

## After it finishes

Download the `.npz` from the version's **Output** tab into the repo:

| JOB | goes to |
|---|---|
| `clip_emotion` | `features/clip_ftpf_emotion_v4_results/` |
| `clip_overall` | `features/clip_ftpf_overall_v4_results/` |
| `qwen4b` | `features/vlm4b_features.npz` |
| `qwen8b` | `features/vlm_features.npz` |

The two qwen files hold every saved layer. Pick the one the paper uses:

```
python src/data/select_vlm_layer.py --vlm features/vlm4b_features.npz \
    --type LT --layer 15 --out features/vlm4b_LT15.npz
python src/data/select_vlm_layer.py --vlm features/vlm_features.npz \
    --type LT --layer 15 --out features/vlm_LT15.npz
```

Then locally:

```
python main.py backbone --backbones clip,clip_ft,clip_ft_emo,qwen4b,qwen8b
```